# 🎬 Asterisk Gateway & Sovereign Agent: Video Orchestrator
### Automatický generátor prezentačního videa pro hackathony
- **Formát:** 1080p Cyberpunk/Aerospace HUD video se synchronizovaným českým neuronovým komentářem
- **Témata:** DO-178C DAL-A (881+ dokázaných kontrol), n8n jako autonomní nervová soustava, 3D WebXR avatar ČZJ, Standard 700
- **Výstup:**  a podklady pro NotebookLM / Veo

In [ ]:
# KROK 1: Instalace potřebných knihoven (pokud ještě nejsou nainstalovány)
!pip install edge-tts moviepy pillow gTTS

In [ ]:
# KROK 2: Definice scén, voiceoveru a metrik
import os, json, asyncio
from pathlib import Path
from PIL import Image, ImageDraw

OUTPUT_DIR = Path("./video_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open("/home/asgard/Plocha/Spolecna_Plocha/video_output/video_manifest.json", encoding="utf-8") as f:
    SCENES = json.load(f)

print(f"Načteno {len(SCENES)} scén pro video.")

In [ ]:
# KROK 3: Generování českého neuronového hlasu (edge-tts)
import edge_tts

async def synthesize_all_audio():
    for scene in SCENES:
        mp3_path = OUTPUT_DIR / f"{scene['id']}.mp3"
        communicate = edge_tts.Communicate(scene["voiceover"], "cs-CZ-AntoninNeural")
        await communicate.save(str(mp3_path))
        print(f"🔊 Zvuk vygenerován: {mp3_path.name}")

await synthesize_all_audio()
print("✅ Všechny audio stopy jsou hotovy!")

In [ ]:
# KROK 4: Spojení slidů a audia do finálního MP4 videa
from moviepy.editor import ImageClip, AudioFileClip, concatenate_videoclips

clips = []
for scene in SCENES:
    img_path = str(OUTPUT_DIR / f"{scene['id']}.png")
    audio_path = str(OUTPUT_DIR / f"{scene['id']}.mp3")
    
    if os.path.exists(audio_path):
        audio_clip = AudioFileClip(audio_path)
        duration = audio_clip.duration + 0.5 # malá pauza mezi scénami
        img_clip = ImageClip(img_path).set_duration(duration).set_audio(audio_clip)
        clips.append(img_clip)
        print(f"🎬 Scéna připravena: {scene['id']} (délka: {duration:.1f}s)")

final_video = concatenate_videoclips(clips, method="compose")
final_path = OUTPUT_DIR / "sovereign_showcase_video.mp4"
final_video.write_videofile(str(final_path), fps=24, codec="libx264", audio_codec="aac")
print(f"🏆 FINÁLNÍ VIDEO ULOŽENO: {final_path}")

In [ ]:
# KROK 5: Přehrání vygenerovaného videa přímo v notebooku
from IPython.display import Video, display
video_file = str(OUTPUT_DIR / "sovereign_showcase_video.mp4")
if os.path.exists(video_file):
    display(Video(video_file, embed=True, width=800))
else:
    print("Video soubor ještě nebyl vygenerován.")